In [ ]:
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from torchinfo import summary

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(256, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))

        x = self.pool2(F.relu(self.conv2(x)))

        # Flatten
        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))

        x = F.relu(self.fc2(x))

        x = self.fc3(x)

        output = F.log_softmax(x, dim=1)
        return output

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(), # Converts PIL Image or numpy.ndarray to FloatTensor
    transforms.Normalize((0.1307,), (0.3081,)) # Mean and std deviation for MNIST
])

In [ ]:
train_dataset = datasets.MNIST('../data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('../data', train=False, download=True, transform=transform)

# Define DataLoaders to batch and shuffle the data
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

In [ ]:
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
else:
    print ("MPS device not found.")

In [ ]:
model = Net().to(mps_device)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001) 
criterion = nn.NLLLoss()

In [ ]:
def train(model, device, train_loader, optimizer, epoch):
    model.train() # Set the model to training mode (e.g., enables dropout if present)
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad() # Zero the gradients before each batch
        output = model(data)  # Forward pass
        loss = criterion(output, target) # Calculate the loss
        loss.backward()       # Backward pass: compute gradients
        optimizer.step()      # Update model parameters
        
        running_loss += loss.item()
        if batch_idx % 100 == 0: # Print training status every 100 batches
            print(f"Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} "
                  f"({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}")
    
    avg_loss = running_loss / len(train_loader)
    print(f'====> Epoch: {epoch} Average training loss: {avg_loss:.4f}')
    return avg_loss
    
def test(model, device, test_loader):
    model.eval() # Set the model to evaluation mode (e.g., disables dropout)
    test_loss = 0
    correct = 0
    with torch.no_grad(): # Disable gradient calculation for evaluation
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item() # Sum up batch loss
            pred = output.argmax(dim=1, keepdim=True) # Get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item() # Count correct predictions

    test_loss /= len(test_loader.dataset) # Average loss per sample

    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} '
          f'({100. * correct / len(test_loader.dataset):.0f}%)\n')
    return test_loss, 100. * correct / len(test_loader.dataset)

In [ ]:
epochs = 10 # Number of training epochs
train_losses = []
test_losses = []
test_accuracies = []

for epoch in range(1, epochs + 1):
    train_loss = train(model, mps_device, train_loader, optimizer, epoch)
    test_loss, test_acc = test(model, mps_device, test_loader)
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

In [ ]:
plt.figure(figsize=(12, 5))

# Plot training and test loss
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), train_losses, label='Training Loss')
plt.plot(range(1, epochs + 1), test_losses, label='Test Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), test_accuracies, label='Test Accuracy', color='orange')
plt.title('Test Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
model.eval()
data, target = next(iter(test_loader))
data, target = data.to(mps_device), target.to(mps_device)
output = model(data)
predictions = output.argmax(dim=1, keepdim=True)


In [ ]:
# Display a few test images with their predictions
fig = plt.figure(figsize=(10, 8))
for i in range(min(16, len(data))): # Display up to 16 images
    ax = fig.add_subplot(4, 4, i + 1, xticks=[], yticks=[])
    # Unnormalize the image for display
    img = data[i].cpu().numpy().squeeze() # Remove batch and channel dim, move to CPU
    img = img * 0.3081 + 0.1307 # Reverse normalization
    ax.imshow(img, cmap='gray')
    ax.set_title(f"True: {target[i].item()}\nPred: {predictions[i].item()}", 
                 color=("green" if predictions[i].item() == target[i].item() else "red"))
plt.tight_layout()
plt.show()

In [ ]:
model_save_path = 'lenet5_mnist_model.pth'
torch.save(model.state_dict(), model_save_path)

In [ ]:
# Assuming 'model' is your trained LeNet5 instance
# Access the state_dict
state_dict = model.state_dict()

c_array_output = []

for name, param in state_dict.items():
    # Convert the PyTorch tensor to a NumPy array, then flatten it
    param_np = param.cpu().numpy().flatten()
    
    # Determine the type (float for weights/biases) and array size
    param_type = "float" # Assuming float for weights and biases
    array_name = name.replace('.', '_') # Replace dots with underscores for C variable names
    array_size = len(param_np)
    
    # Format the array elements
    # Using 6 decimal places for precision
    param_str = ", ".join([f"{x:.6f}f" for x in param_np])
    
    # Get the original shape for commenting
    original_shape_str = str(param.shape).replace('torch.Size', '')
    
    # Construct the C array definition string
    c_array_definition = (
        f"// Original shape: {original_shape_str}\n"
        f"const {param_type} {array_name}[{array_size}] = {{\n"
        f"    {param_str}\n"
        f"}};\n"
    )
    c_array_output.append(c_array_definition)

# Print all C array definitions
for c_array in c_array_output:
    print(c_array)

print("\n--- C Array Export Complete ---")
print("You can copy the above C array definitions into your C/C++ project.")
print("Remember to implement the forward pass logic in C to use these weights.")

In [ ]:
model = Net()
summary(model, input_size=(1, 1, 28, 28))